In [1]:
import os
from pathlib import Path

import autoroot  # noqa: F401
import polars as pl
from sklearn.model_selection import train_test_split

df = pl.read_csv("hf://datasets/dllllb/rosbank-churn/train.csv.gz").filter(currency=810)

In [2]:
# Frequency-encode MCCs

mcc_renamer = dict(
    df.select(pl.col("MCC").value_counts(sort=True))
    .unnest("MCC")
    .with_row_index(offset=1)
    .select("MCC", "index")
    .iter_rows()
)

In [3]:
df_processed = (
    df.with_columns(
        datetime=pl.col("TRDATETIME").str.to_datetime("%d%b%y:%T"),
        MCC=pl.col("MCC").replace(mcc_renamer),
        amount=pl.col("amount").abs().log1p(),
        target="target_flag",
    )
    .filter(
        (pl.col("datetime").max() - pl.col("datetime").min()).over("cl_id") > pl.duration(weeks=1),
        pl.len().over("cl_id") > 5,
    )
    .with_columns(
        time=(pl.col("datetime") - pl.col("datetime").min()).over("cl_id")
        / (pl.col("datetime").max() - pl.col("datetime").min()).over("cl_id").median()
    )
    .drop(
        "PERIOD",
        "datetime",
        "channel_type",
        "currency",
        "trx_category",
        "TRDATETIME",
        "target_flag",
        "target_sum",
    )
    .group_by("cl_id")
    .agg(
        pl.exclude("target").sort_by("time"),
        pl.col("target").first(),
    )
)


In [4]:
df_trainval, df_test = train_test_split(
    df_processed, test_size=1000, random_state=42, stratify=df_processed["target"]
)
df_train, df_val = train_test_split(
    df_trainval, test_size=100, stratify=df_trainval["target"]
)

In [5]:
df_train.write_parquet(
    Path(os.environ["DATA_DIR"]) / "preprocessed/churn_train.parquet"
)
df_val.write_parquet(Path(os.environ["DATA_DIR"]) / "preprocessed/churn_val.parquet")
df_test.write_parquet(Path(os.environ["DATA_DIR"]) / "preprocessed/churn_test.parquet")
